# 07. SBERT + FAISS Vector Retrieval

Sentence-BERT 임베딩과 FAISS 벡터 검색으로 민원 QA 검색 성능을 평가합니다.
BM25 baseline(06)과 비교하여 의미 기반 검색의 개선 효과를 측정합니다.

| Item | Detail |
|------|--------|
| Task | QA 검색: 질문 -> 가장 유사한 답변 검색 |
| Models | `jhgan/ko-sbert-nli`, `BM-K/KoSimCSE-roberta` |
| Index | FAISS IndexFlatIP (cosine similarity) |
| Metrics | Recall@k, MRR@k (k=1,3,5,10,20) |
| Baseline | BM25 (06_bm25_retrieval) |
| Environment | Kaggle T4 x2 GPU |

---
## 1. Environment Setup

In [ ]:
%%capture
!pip install -q sentence-transformers faiss-gpu plotly kaleido

In [ ]:
import os, json, time, warnings, pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'
warnings.filterwarnings('ignore')

import torch
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"  cuDNN benchmark = True, float32 matmul precision = medium")

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

MODELS_DIR = os.path.join(OUT_DIR, 'models/embedding')
INDEX_DIR = os.path.join(OUT_DIR, 'data/vectordb/faiss_index')
RESULTS_DIR = os.path.join(OUT_DIR, 'results')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load BM25 baseline
bm25_results = None
bm25_path = os.path.join(RESULTS_DIR, 'retrieval_bm25_results.json')
if os.path.exists(bm25_path):
    with open(bm25_path) as f:
        bm25_results = json.load(f)
    print("BM25 baseline loaded")
else:
    print("BM25 baseline not found -- will skip comparison where needed")

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data: {DATA_DIR}")
print(f"Models: {MODELS_DIR}")
print(f"Index: {INDEX_DIR}")

---
## 2. QA Data Loading

In [ ]:
qa_path = os.path.join(DATA_DIR, 'qa_pairs.parquet')
qa_df = pd.read_parquet(qa_path)

print(f"QA pairs: {len(qa_df):,}")
print(f"Columns: {list(qa_df.columns)}")
qa_df.head(3)

In [ ]:
# Train / Test split for evaluation
# Use same random seed as BM25 for comparable test set
from sklearn.model_selection import train_test_split

corpus_df, test_df = train_test_split(qa_df, test_size=0.1, random_state=42)
corpus_df = corpus_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

corpus_questions = corpus_df['question'].tolist()
corpus_answers = corpus_df['answer'].tolist()
test_questions = test_df['question'].tolist()
test_answers = test_df['answer'].tolist()

# Domain info for per-domain evaluation
has_domain = 'domain' in test_df.columns
if has_domain:
    test_domains = test_df['domain'].tolist()
    domain_list = sorted(test_df['domain'].unique())
    print(f"Domains: {len(domain_list)}")

print(f"Corpus: {len(corpus_df):,} / Test: {len(test_df):,}")
print(f"Avg question length: {qa_df['question'].str.len().mean():.0f} chars")
print(f"Avg answer length:   {qa_df['answer'].str.len().mean():.0f} chars")

---
## 3. Embedding Model Loading

| Model | Base | Dim | Note |
|-------|------|-----|------|
| `jhgan/ko-sbert-nli` | KoBERT | 768 | Korean SBERT, NLI fine-tuned |
| `BM-K/KoSimCSE-roberta` | RoBERTa | 768 | SimCSE contrastive learning |

In [ ]:
MODEL_CONFIGS = [
    {'name': 'ko-sbert-nli',        'hf_id': 'jhgan/ko-sbert-nli'},
    {'name': 'KoSimCSE-roberta',     'hf_id': 'BM-K/KoSimCSE-roberta'},
]

models = {}
for cfg in MODEL_CONFIGS:
    print(f"Loading {cfg['name']} ({cfg['hf_id']}) ...")
    t0 = time.time()
    m = SentenceTransformer(cfg['hf_id'])
    load_time = time.time() - t0
    dim = m.get_sentence_embedding_dimension()
    models[cfg['name']] = m
    print(f"  Embedding dim: {dim} | Load time: {load_time:.1f}s")
    print(f"  Max seq length: {m.max_seq_length}")

print(f"\nModels loaded: {list(models.keys())}")

---
## 4. Corpus Vectorization

In [ ]:
corpus_embeddings = {}
test_embeddings = {}
encoding_stats = []

for model_name, model in models.items():
    print(f"\n=== {model_name} ===")

    # Encode corpus
    print(f"  Encoding corpus ({len(corpus_questions):,} docs) ...")
    t0 = time.time()
    c_emb = model.encode(
        corpus_questions,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    corpus_time = time.time() - t0
    corpus_speed = len(corpus_questions) / corpus_time

    # Encode test queries
    print(f"  Encoding queries ({len(test_questions):,}) ...")
    t0 = time.time()
    q_emb = model.encode(
        test_questions,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    query_time = time.time() - t0

    corpus_embeddings[model_name] = c_emb.astype(np.float32)
    test_embeddings[model_name] = q_emb.astype(np.float32)

    mem_mb = c_emb.nbytes / 1024 / 1024
    stat = {
        'model': model_name,
        'dim': c_emb.shape[1],
        'corpus_time_s': round(corpus_time, 1),
        'corpus_speed': round(corpus_speed, 0),
        'query_time_s': round(query_time, 1),
        'memory_mb': round(mem_mb, 1),
    }
    encoding_stats.append(stat)
    print(f"  Shape: {c_emb.shape} | Speed: {corpus_speed:.0f} samples/s")
    print(f"  Memory: {mem_mb:.1f} MB | Corpus: {corpus_time:.1f}s | Queries: {query_time:.1f}s")

pd.DataFrame(encoding_stats)

---
## 5. FAISS Index Construction

IndexFlatIP (Inner Product)를 사용합니다.  
임베딩이 L2-normalize 되어 있으므로 IP = cosine similarity 입니다.

In [ ]:
faiss_indices = {}

for model_name, emb in corpus_embeddings.items():
    print(f"\n=== Building FAISS index for {model_name} ===")
    dim = emb.shape[1]

    # IndexFlatIP: exact inner product search
    index = faiss.IndexFlatIP(dim)

    # Use GPU if available
    try:
        gpu_res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(gpu_res, 0, index)
        print(f"  GPU index enabled")
    except Exception:
        print(f"  CPU index (GPU not available)")

    t0 = time.time()
    index.add(emb)
    build_time = time.time() - t0

    faiss_indices[model_name] = index
    print(f"  Vectors: {index.ntotal:,}")
    print(f"  Dimension: {dim}")
    print(f"  Build time: {build_time:.2f}s")

    # Sanity check: search with first query
    D, I = index.search(test_embeddings[model_name][:1], 5)
    print(f"  Sanity check top-5 scores: {D[0].tolist()}")

---
## 6. Retrieval Evaluation

BM25와 동일한 프로토콜: Recall@k, MRR@k (k=1,3,5,10,20)

In [ ]:
def evaluate_faiss_retrieval(index, query_embeddings, test_answers, corpus_answers,
                              k_values=[1, 3, 5, 10, 20]):
    """Evaluate FAISS retrieval with Recall@k and MRR@k."""
    max_k = max(k_values)
    D, I = index.search(query_embeddings, max_k)

    results = {k: {'recall': [], 'mrr': []} for k in k_values}

    for i, gt_answer in enumerate(test_answers):
        for k in k_values:
            top_k_indices = I[i][:k]
            retrieved = [corpus_answers[idx] for idx in top_k_indices]

            hit = gt_answer in retrieved
            results[k]['recall'].append(1.0 if hit else 0.0)

            rr = 0.0
            for rank, ans in enumerate(retrieved, 1):
                if ans == gt_answer:
                    rr = 1.0 / rank
                    break
            results[k]['mrr'].append(rr)

    return {
        k: {
            'recall': np.mean(v['recall']),
            'mrr': np.mean(v['mrr']),
        }
        for k, v in results.items()
    }


def evaluate_faiss_per_domain(index, query_embeddings, test_answers, corpus_answers,
                               test_domains, domain_list, k_values=[1, 3, 5, 10, 20]):
    """Per-domain evaluation."""
    max_k = max(k_values)
    D, I = index.search(query_embeddings, max_k)

    domain_results = {d: {k: {'recall': [], 'mrr': []} for k in k_values} for d in domain_list}

    for i, gt_answer in enumerate(test_answers):
        d = test_domains[i]
        for k in k_values:
            top_k_indices = I[i][:k]
            retrieved = [corpus_answers[idx] for idx in top_k_indices]

            hit = gt_answer in retrieved
            domain_results[d][k]['recall'].append(1.0 if hit else 0.0)

            rr = 0.0
            for rank, ans in enumerate(retrieved, 1):
                if ans == gt_answer:
                    rr = 1.0 / rank
                    break
            domain_results[d][k]['mrr'].append(rr)

    return {
        d: {
            k: {
                'recall': np.mean(v['recall']) if v['recall'] else 0.0,
                'mrr': np.mean(v['mrr']) if v['mrr'] else 0.0,
            }
            for k, v in kv.items()
        }
        for d, kv in domain_results.items()
    }

print("Evaluation functions defined.")

In [ ]:
K_VALUES = [1, 3, 5, 10, 20]
all_eval_results = {}
all_domain_results = {}

for model_name in models:
    print(f"\n=== Evaluating {model_name} ===")
    t0 = time.time()
    res = evaluate_faiss_retrieval(
        faiss_indices[model_name],
        test_embeddings[model_name],
        test_answers,
        corpus_answers,
        k_values=K_VALUES,
    )
    eval_time = time.time() - t0
    all_eval_results[model_name] = res

    for k in K_VALUES:
        print(f"  Recall@{k:<3} = {res[k]['recall']:.4f}   MRR@{k:<3} = {res[k]['mrr']:.4f}")
    print(f"  Eval time: {eval_time:.1f}s")

    # Per-domain evaluation
    if has_domain:
        dom_res = evaluate_faiss_per_domain(
            faiss_indices[model_name],
            test_embeddings[model_name],
            test_answers,
            corpus_answers,
            test_domains,
            domain_list,
            k_values=K_VALUES,
        )
        all_domain_results[model_name] = dom_res

In [ ]:
# Summary table
summary_rows = []
for model_name, res in all_eval_results.items():
    row = {'model': model_name}
    for k in K_VALUES:
        row[f'Recall@{k}'] = round(res[k]['recall'], 4)
        row[f'MRR@{k}'] = round(res[k]['mrr'], 4)
    summary_rows.append(row)

# Add BM25 if available
if bm25_results is not None:
    row = {'model': 'BM25 (baseline)'}
    bm25_metrics = bm25_results.get('metrics', bm25_results)
    for k in K_VALUES:
        k_str = str(k)
        row[f'Recall@{k}'] = bm25_metrics.get(f'recall_at_{k}', bm25_metrics.get(k_str, {}).get('recall', None))
        row[f'MRR@{k}'] = bm25_metrics.get(f'mrr_at_{k}', bm25_metrics.get(k_str, {}).get('mrr', None))
    summary_rows.insert(0, row)

summary_df = pd.DataFrame(summary_rows)
summary_df

---
## 7. BM25 vs SBERT Comparison

In [ ]:
# ---- Grouped bar chart: Recall@k ----
recall_cols = [c for c in summary_df.columns if c.startswith('Recall@')]

colors_map = {
    'BM25 (baseline)': '#ef5350',
    'ko-sbert-nli': '#42a5f5',
    'KoSimCSE-roberta': '#66bb6a',
}

fig = go.Figure()
for _, row in summary_df.iterrows():
    name = row['model']
    vals = [row[c] for c in recall_cols]
    fig.add_trace(go.Bar(
        x=recall_cols, y=vals, name=name,
        marker_color=colors_map.get(name, '#999'),
        text=[f'{v:.3f}' if v is not None else '' for v in vals],
        textposition='outside',
    ))

fig.update_layout(
    title='Recall@k Comparison: BM25 vs SBERT Models',
    xaxis_title='Metric', yaxis_title='Score',
    yaxis_range=[0, 1.1], barmode='group',
    width=900, height=500, margin=dict(t=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# ---- MRR@k grouped bar chart ----
mrr_cols = [c for c in summary_df.columns if c.startswith('MRR@')]

fig = go.Figure()
for _, row in summary_df.iterrows():
    name = row['model']
    vals = [row[c] for c in mrr_cols]
    fig.add_trace(go.Bar(
        x=mrr_cols, y=vals, name=name,
        marker_color=colors_map.get(name, '#999'),
        text=[f'{v:.3f}' if v is not None else '' for v in vals],
        textposition='outside',
    ))

fig.update_layout(
    title='MRR@k Comparison: BM25 vs SBERT Models',
    xaxis_title='Metric', yaxis_title='Score',
    yaxis_range=[0, 1.1], barmode='group',
    width=900, height=500, margin=dict(t=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# ---- Recall@k curve (line chart) ----
fig = go.Figure()
for _, row in summary_df.iterrows():
    name = row['model']
    vals = [row[f'Recall@{k}'] for k in K_VALUES]
    fig.add_trace(go.Scatter(
        x=K_VALUES, y=vals, mode='lines+markers', name=name,
        line=dict(color=colors_map.get(name, '#999'), width=3),
        marker=dict(size=10),
    ))

fig.update_layout(
    title='Recall@k Curve: BM25 vs SBERT Models',
    xaxis_title='k', yaxis_title='Recall@k',
    yaxis_range=[0, 1.05], xaxis=dict(tickvals=K_VALUES),
    width=800, height=500, margin=dict(t=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# ---- Per-domain Recall@5 heatmap ----
if has_domain and all_domain_results:
    # Build heatmap data: rows=domains, cols=methods
    method_names = list(all_domain_results.keys())
    z_data = []
    for d in domain_list:
        row_vals = []
        for mn in method_names:
            row_vals.append(all_domain_results[mn][d][5]['recall'])
        z_data.append(row_vals)

    fig = go.Figure(go.Heatmap(
        z=z_data,
        x=method_names,
        y=domain_list,
        colorscale='Viridis',
        text=[[f'{v:.3f}' for v in row] for row in z_data],
        texttemplate='%{text}',
        textfont=dict(size=11),
    ))
    fig.update_layout(
        title='Per-Domain Recall@5 Heatmap',
        xaxis_title='Model', yaxis_title='Domain',
        width=700, height=max(400, 30 * len(domain_list)),
        margin=dict(t=60),
    )
    fig.show(renderer='iframe')
else:
    print("Per-domain evaluation skipped (no domain column).")

---
## 8. Qualitative Analysis

Sample queries: BM25 top-5 vs SBERT top-5 결과 비교

In [ ]:
# Pick the best SBERT model for qualitative comparison
best_sbert_name = max(all_eval_results, key=lambda n: all_eval_results[n][5]['recall'])
best_index = faiss_indices[best_sbert_name]
best_q_emb = test_embeddings[best_sbert_name]

print(f"Best SBERT model: {best_sbert_name}")
print(f"Recall@5: {all_eval_results[best_sbert_name][5]['recall']:.4f}")

In [ ]:
NUM_SAMPLES = 5
np.random.seed(42)
sample_indices = np.random.choice(len(test_questions), size=NUM_SAMPLES, replace=False)

# SBERT results
D_sbert, I_sbert = best_index.search(best_q_emb[sample_indices], 5)

from IPython.display import display, HTML

html_parts = []
html_parts.append('<h3>Qualitative Comparison: SBERT Top-5 Retrieval</h3>')

for si, idx in enumerate(sample_indices):
    query = test_questions[idx]
    gt = test_answers[idx]

    html_parts.append(f'<div style="margin:20px 0; padding:15px; border:1px solid #ddd; border-radius:8px;">')
    html_parts.append(f'<b>Query {si+1}:</b> {query}<br>')
    html_parts.append(f'<b>Ground Truth:</b> <span style="color:#2e7d32;">{gt[:200]}...</span><br><br>')

    html_parts.append(f'<b>{best_sbert_name} Top-5:</b>')
    html_parts.append('<ol>')
    for rank in range(5):
        ret_idx = I_sbert[si][rank]
        score = D_sbert[si][rank]
        ans = corpus_answers[ret_idx]
        is_hit = (ans == gt)
        color = '#2e7d32' if is_hit else '#333'
        marker = ' [HIT]' if is_hit else ''
        html_parts.append(
            f'<li style="color:{color};">'
            f'(score={score:.4f}) {ans[:150]}...{marker}</li>'
        )
    html_parts.append('</ol></div>')

display(HTML(''.join(html_parts)))

---
## 9. Save Model + Index + Results

In [ ]:
# ---- Save best SentenceTransformer model ----
best_model_path = os.path.join(MODELS_DIR, best_sbert_name)
models[best_sbert_name].save(best_model_path)
print(f"Best model saved: {best_model_path}")

# ---- Save FAISS index (CPU version for portability) ----
for model_name, index in faiss_indices.items():
    idx_path = os.path.join(INDEX_DIR, f'{model_name}.index')
    # Convert GPU index back to CPU for saving
    try:
        cpu_index = faiss.index_gpu_to_cpu(index)
    except Exception:
        cpu_index = index
    faiss.write_index(cpu_index, idx_path)
    print(f"FAISS index saved: {idx_path}")

# Also save primary index as index.faiss
primary_idx_path = os.path.join(INDEX_DIR, 'index.faiss')
try:
    cpu_best = faiss.index_gpu_to_cpu(faiss_indices[best_sbert_name])
except Exception:
    cpu_best = faiss_indices[best_sbert_name]
faiss.write_index(cpu_best, primary_idx_path)
print(f"Primary index saved: {primary_idx_path}")

In [ ]:
# ---- Save document metadata for LangChain compatibility ----
doc_metadata = {
    'corpus_questions': corpus_questions,
    'corpus_answers': corpus_answers,
    'model_name': best_sbert_name,
    'embedding_dim': int(corpus_embeddings[best_sbert_name].shape[1]),
    'corpus_size': len(corpus_questions),
}
if has_domain:
    doc_metadata['corpus_domains'] = corpus_df['domain'].tolist()

meta_path = os.path.join(INDEX_DIR, 'doc_metadata.pkl')
with open(meta_path, 'wb') as f:
    pickle.dump(doc_metadata, f)
print(f"Doc metadata saved: {meta_path}")

In [ ]:
# ---- Save retrieval_sbert_results.json ----
best_res = all_eval_results[best_sbert_name]

experiments = []
for model_name, res in all_eval_results.items():
    exp = {'model': model_name}
    for k in K_VALUES:
        exp[f'recall_at_{k}'] = round(res[k]['recall'], 4)
        exp[f'mrr_at_{k}'] = round(res[k]['mrr'], 4)
    experiments.append(exp)

sbert_output = {
    'experiments': experiments,
    'best_model': best_sbert_name,
    'recall_at_5': round(best_res[5]['recall'], 4),
    'mrr': round(best_res[5]['mrr'], 4),
    'corpus_size': len(corpus_questions),
    'embedding_dim': int(corpus_embeddings[best_sbert_name].shape[1]),
    'encoding_stats': encoding_stats,
}

# Comparison with BM25
if bm25_results is not None:
    bm25_m = bm25_results.get('metrics', bm25_results)
    bm25_r5 = bm25_m.get('recall_at_5', bm25_m.get('5', {}).get('recall', 0.0))
    sbert_r5 = best_res[5]['recall']
    if bm25_r5 and bm25_r5 > 0:
        improvement = (sbert_r5 - bm25_r5) / bm25_r5 * 100
    else:
        improvement = None
    sbert_output['comparison_with_bm25'] = {
        'bm25_recall_at_5': bm25_r5,
        'sbert_recall_at_5': round(sbert_r5, 4),
        'improvement': f'+{improvement:.0f}%' if improvement is not None else 'N/A',
    }

results_path = os.path.join(RESULTS_DIR, 'retrieval_sbert_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(sbert_output, f, ensure_ascii=False, indent=2)

print(f"Results saved: {results_path}")
print(json.dumps(sbert_output, ensure_ascii=False, indent=2))

In [ ]:
# ---- Base64 download helpers ----
import base64, zipfile, io
from IPython.display import display, HTML


def create_download_link(filepath, filename=None):
    """Create an HTML download link for a single file."""
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (
        f'<a href="data:application/octet-stream;base64,{b64}" '
        f'download="{filename}">Download: {filename} ({size_mb:.1f} MB)</a>'
    )
    display(HTML(href))


def create_zip_download(file_dict, zip_name="model_artifacts.zip"):
    """Create a zip archive and show download link."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for arcname, filepath in file_dict.items():
            zf.write(filepath, arcname)
    buffer.seek(0)
    data = buffer.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (
        f'<a href="data:application/octet-stream;base64,{b64}" '
        f'download="{zip_name}">Download: {zip_name} ({size_mb:.1f} MB)</a>'
    )
    display(HTML(href))


print("Download links:")

In [ ]:
# Results JSON
create_download_link(results_path)

# FAISS index
create_download_link(primary_idx_path)

# Doc metadata
create_download_link(meta_path)

# Model + Index zip (may be large)
import glob as _glob

zip_files = {}
# FAISS index files
for fp in _glob.glob(os.path.join(INDEX_DIR, '*')):
    zip_files[f'faiss_index/{os.path.basename(fp)}'] = fp
# Best model files
for fp in _glob.glob(os.path.join(best_model_path, '**', '*'), recursive=True):
    if os.path.isfile(fp):
        rel = os.path.relpath(fp, MODELS_DIR)
        zip_files[f'embedding/{rel}'] = fp

if zip_files:
    create_zip_download(zip_files, zip_name="sbert_faiss_artifacts.zip")
else:
    print("No files to zip.")

---
## 10. Summary

In [ ]:
# ---- Progressive improvement visualization ----
method_order = []
recall5_values = []
mrr5_values = []
bar_colors = []

if bm25_results is not None:
    bm25_m = bm25_results.get('metrics', bm25_results)
    bm25_r5 = bm25_m.get('recall_at_5', bm25_m.get('5', {}).get('recall', 0.0))
    bm25_mrr5 = bm25_m.get('mrr_at_5', bm25_m.get('5', {}).get('mrr', 0.0))
    method_order.append('BM25')
    recall5_values.append(bm25_r5)
    mrr5_values.append(bm25_mrr5)
    bar_colors.append('#ef5350')

for model_name in ['ko-sbert-nli', 'KoSimCSE-roberta']:
    if model_name in all_eval_results:
        method_order.append(model_name)
        recall5_values.append(all_eval_results[model_name][5]['recall'])
        mrr5_values.append(all_eval_results[model_name][5]['mrr'])
        bar_colors.append('#42a5f5' if model_name == 'ko-sbert-nli' else '#66bb6a')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Recall@5 Progression', 'MRR@5 Progression'],
)

fig.add_trace(go.Bar(
    x=method_order, y=recall5_values,
    marker_color=bar_colors,
    text=[f'{v:.4f}' for v in recall5_values],
    textposition='outside', showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=method_order, y=mrr5_values,
    marker_color=bar_colors,
    text=[f'{v:.4f}' for v in mrr5_values],
    textposition='outside', showlegend=False,
), row=1, col=2)

fig.update_yaxes(range=[0, 1.1])
fig.update_layout(
    title='Progressive Improvement: BM25 -> ko-sbert-nli -> KoSimCSE-roberta',
    width=900, height=450, margin=dict(t=80),
)
fig.show(renderer='iframe')

In [ ]:
# ---- Radar chart: BM25 vs Best SBERT across k values ----
categories = [f'Recall@{k}' for k in K_VALUES] + [f'MRR@{k}' for k in K_VALUES]

best_vals = []
for k in K_VALUES:
    best_vals.append(all_eval_results[best_sbert_name][k]['recall'])
for k in K_VALUES:
    best_vals.append(all_eval_results[best_sbert_name][k]['mrr'])

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=best_vals + [best_vals[0]],
    theta=categories + [categories[0]],
    fill='toself', name=best_sbert_name,
    line=dict(color='#66bb6a'),
))

if bm25_results is not None:
    bm25_vals = []
    bm25_m = bm25_results.get('metrics', bm25_results)
    for k in K_VALUES:
        bm25_vals.append(bm25_m.get(f'recall_at_{k}', bm25_m.get(str(k), {}).get('recall', 0.0)))
    for k in K_VALUES:
        bm25_vals.append(bm25_m.get(f'mrr_at_{k}', bm25_m.get(str(k), {}).get('mrr', 0.0)))
    fig.add_trace(go.Scatterpolar(
        r=bm25_vals + [bm25_vals[0]],
        theta=categories + [categories[0]],
        fill='toself', name='BM25',
        line=dict(color='#ef5350'),
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f'Radar: BM25 vs {best_sbert_name}',
    width=700, height=600, margin=dict(t=80),
)
fig.show(renderer='iframe')

In [ ]:
print("=" * 60)
print("     07. SBERT + FAISS Vector Retrieval -- Summary")
print("=" * 60)
print()
print(f"  Best model:     {best_sbert_name}")
print(f"  Embedding dim:  {corpus_embeddings[best_sbert_name].shape[1]}")
print(f"  Corpus size:    {len(corpus_questions):,}")
print()
print("  Retrieval Performance:")
for k in K_VALUES:
    r = all_eval_results[best_sbert_name][k]['recall']
    m = all_eval_results[best_sbert_name][k]['mrr']
    print(f"    Recall@{k:<3} = {r:.4f}   MRR@{k:<3} = {m:.4f}")
print()
if bm25_results is not None and 'comparison_with_bm25' in sbert_output:
    comp = sbert_output['comparison_with_bm25']
    print(f"  vs BM25 (Recall@5): {comp['bm25_recall_at_5']:.4f} -> {comp['sbert_recall_at_5']:.4f} ({comp['improvement']})")
    print()
print(f"  Artifacts:")
print(f"    Model  -> {MODELS_DIR}/{best_sbert_name}/")
print(f"    Index  -> {INDEX_DIR}/")
print(f"    Results -> {results_path}")
print()
print("  Next -> 08_rag_pipeline")
print("=" * 60)

---
## 11. 임베딩 모델 및 인덱스 선택 근거

### 모델 선택

| Model | Base | Dim | Training | Why Selected |
|-------|------|-----|----------|--------------|
| `jhgan/ko-sbert-nli` | KoBERT | 768 | NLI (자연어 추론) fine-tuning | 한국어 SBERT 중 가장 널리 검증됨. NLI 학습으로 문장 간 의미 유사도에 특화 |
| `BM-K/KoSimCSE-roberta` | RoBERTa | 768 | SimCSE (대조 학습) | Contrastive learning으로 동일 의미의 다른 표현을 가까이 매핑. 유의어 처리에 강점 |

두 모델을 비교한 이유: NLI 기반(SBERT)과 Contrastive 기반(SimCSE)은 학습 목적이 달라 **검색 task에서 상보적 강약점**을 가질 수 있음.

### FAISS IndexFlatIP 선택 근거

| Alternative | Pros | Cons | Decision |
|------------|------|------|----------|
| **IndexFlatIP** | 100% 정확도 (brute-force), 구현 단순 | O(n) 검색 시간 | ✅ 선택: 57K 규모에서는 GPU로 1ms 미만 |
| IndexIVFFlat | 근사 검색으로 속도 향상 | nprobe 튜닝 필요, recall 손실 | 10만 건 이하에서는 과도한 복잡성 |
| IndexHNSW | 높은 recall + 빠른 검색 | 메모리 2~3배, 빌드 시간 | 100만 건 이상에서 고려 |

**결론**: 현재 코퍼스 규모(~57K)에서는 IndexFlatIP가 정확도-속도-구현복잡성 모든 면에서 최적.
L2-normalized 벡터에 대해 Inner Product = Cosine Similarity이므로 별도 정규화 불필요.

In [ ]:
# === BM25 → SBERT Recall@k 개선율 정량 분석 ===
print("=" * 70)
print("  BM25 → SBERT+FAISS 검색 성능 개선율")
print("=" * 70)

if bm25_results is not None:
    bm25_m = bm25_results.get('metrics', bm25_results)
    best_res = all_eval_results[best_sbert_name]

    print(f"\n  Best SBERT model: {best_sbert_name}")
    print(f"\n  {'Metric':<14} {'BM25':>10} {'SBERT':>10} {'Delta':>10} {'Relative':>10}")
    print(f"  {'-'*56}")

    for k in K_VALUES:
        bm25_r = bm25_m.get(f'recall_at_{k}', 0)
        sbert_r = best_res[k]['recall']
        delta = sbert_r - bm25_r
        rel = (delta / bm25_r * 100) if bm25_r > 0 else 0
        print(f"  {'Recall@' + str(k):<14} {bm25_r:>10.4f} {sbert_r:>10.4f} {delta:>+10.4f} {rel:>+9.1f}%")

    print()
    for k in K_VALUES:
        bm25_mrr = bm25_m.get(f'mrr_at_{k}', 0)
        sbert_mrr = best_res[k]['mrr']
        delta = sbert_mrr - bm25_mrr
        rel = (delta / bm25_mrr * 100) if bm25_mrr > 0 else 0
        print(f"  {'MRR@' + str(k):<14} {bm25_mrr:>10.4f} {sbert_mrr:>10.4f} {delta:>+10.4f} {rel:>+9.1f}%")
else:
    print("  BM25 results not available for comparison.")

print("=" * 70)

---
## 12. Cross-Stage 연결: Retrieval → Generation

**검색 품질이 생성 품질의 상한선을 결정합니다.**

### Retrieval → RAG Pipeline 연결 구조

```
Query → [SBERT Encoder] → [FAISS Search] → Top-k Documents → [LLM Prompt] → Answer
```

1. **Top-k 선택이 답변 품질을 좌우**: Recall@5가 높을수록 LLM이 참조할 "정답" 문서가 포함될 확률 증가
2. **Score threshold**: FAISS similarity score가 낮은 문서는 noise → LLM hallucination 유발 가능
3. **도메인 필터링**: 05에서 학습한 분류기로 도메인을 먼저 예측 → 해당 도메인 QA만 검색 → 정밀도 향상

### 다음 단계 (08_rag_pipeline) 핵심 질문

- Zero-shot(검색 없이) vs RAG(검색+생성): 실제로 검색이 답변 품질을 얼마나 높이는가?
- top_k=3 vs 5 vs 10: 컨텍스트 양과 품질의 트레이드오프
- 검색 오류(wrong document retrieved)가 생성에 미치는 영향 분석

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 결과 + 모델 + 인덱스 심볼릭 링크
    upload_targets = [results_path, primary_idx_path, meta_path]
    for src in upload_targets:
        if os.path.exists(src):
            dst = os.path.join(UPLOAD_DIR, os.path.basename(src))
            if os.path.exists(dst):
                os.remove(dst)
            os.symlink(src, dst)

    # Best model 디렉토리 링크
    best_model_upload = os.path.join(UPLOAD_DIR, best_sbert_name)
    if os.path.exists(best_model_upload):
        import shutil
        shutil.rmtree(best_model_upload)
    os.symlink(best_model_path, best_model_upload)

    meta = {
        "title": "civilcomplaint-sbert-faiss",
        "id": "kukass/civilcomplaint-sbert-faiss",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("✅ Kaggle 데이터셋 업로드 완료: civilcomplaint-sbert-faiss")
else:
    print("ℹ️ 로컬 환경 — Kaggle 업로드 건너뜀")